In [1]:
import numpy as np

In [2]:
# For reproducibility
np.random.seed(2505)

## Dataset

In [3]:
# Generate the small dataset to test
N = 20
D = 6
mean = 2
std = 0.01
V = 0.001 # from noise

In [4]:
# True vector
true_w = np.random.normal(mean, std, size=(D, 1))

In [5]:
print("True weight:", true_w, "\nshape:", true_w.shape)

True weight: [[2.01376019]
 [1.99658355]
 [1.98952207]
 [2.00374753]
 [1.99752854]
 [2.00539425]] 
shape: (6, 1)


### Training data

In [6]:
# Training data
X = np.random.normal(mean, std, size=(N, D))
y = X @ true_w + np.random.normal(mean, std, size=(N, 1))

In [7]:
print("Training sample shape: ", X.shape)
print("Training label shape: ", y.shape)

Training sample shape:  (20, 6)
Training label shape:  (20, 1)


### Prior knowledge

In [8]:
m0 = np.random.normal(mean, std, size=(D, 1))
Cov0 = np.diag(np.random.normal(mean, std, size=(D,)))

In [9]:
print("Prior mean: \n", m0)
print("Piror covariance matrix :\n", Cov0)

Prior mean: 
 [[2.00929185]
 [1.9957354 ]
 [1.99231504]
 [2.00746721]
 [1.99147964]
 [2.0107031 ]]
Piror covariance matrix :
 [[2.00334839 0.         0.         0.         0.         0.        ]
 [0.         2.00656053 0.         0.         0.         0.        ]
 [0.         0.         2.00132431 0.         0.         0.        ]
 [0.         0.         0.         2.00559474 0.         0.        ]
 [0.         0.         0.         0.         1.99800207 0.        ]
 [0.         0.         0.         0.         0.         1.98930085]]


## Classical solvers

### Closed-form solution

$$\mathbf{w}^{*} = \left(\mathbf{X}^\top\mathbf{X}+V\Sigma_{0}^{-1}\right)^{-1}\left(\mathbf{X}^\top\mathbf{y} + V\Sigma_0^{-1}\mathbf{m}_{0}\right)$$

In [10]:
gram_matrix = X.T @ X + V*np.linalg.inv(Cov0)
b = X.T @ y + V*np.linalg.inv(Cov0) @ m0
w_star = np.linalg.inv(gram_matrix) @ b

In [11]:
print("Closed form solution: ", w_star)

Closed form solution:  [[2.08743299]
 [2.19603303]
 [2.18537865]
 [2.29312828]
 [2.03616759]
 [2.2093816 ]]


In [12]:
# measure relative euclidean distance between the closed-form to the true one
l2_dis = np.linalg.norm(w_star - true_w)/np.linalg.norm(true_w)
# measure the cosine similarity between the closed-form to the true one
cos_dis = 1-np.dot(w_star.T, true_w)/(np.linalg.norm(w_star)*np.linalg.norm(true_w))

In [13]:
print("Relative Euclidean distance between the closed-form solution to the true solution: ", l2_dis)
print("Cosine distance between the closed-form solution to the true solution: ", cos_dis)

Relative Euclidean distance between the closed-form solution to the true solution:  0.09358162569275746
Cosine distance between the closed-form solution to the true solution:  [[0.00077114]]


### Gradient descent training

In [14]:
MAX_ITERATION = 200
BATCH_SIZE = 2

In [15]:
total_batch = N//2

for i in range(MAX_ITERATION):
    pass

## Proposed: Variational Quantum Bayessian Algorithm

In [16]:
!pip install qiskit==2.3 -q
!pip install qiskit_aer -q
!pip install qiskit_ibm_runtime -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.6/378.6 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 8.5 MB/s eta 0:00:00


In [17]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
from qiskit.circuit.library import StatePreparation
from qiskit.circuit.library import efficient_su2
from qiskit_aer import AerSimulator
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from scipy.optimize import minimize

In [18]:
SHOTS = 2**10
BATCH_SIZE = 2
OPTIMIZER = "COBYLA"
NUM_QUBIT = int(np.ceil(np.log2(X.shape[1])))
REPS = 1
OPTIMIZATION_LEVEL = 1
backend = AerSimulator()
pm = generate_preset_pass_manager(optimization_level=OPTIMIZATION_LEVEL, backend=backend)
sampler = Sampler(mode=backend)
SHOTS = 2**10

In [19]:
def pad_to_pow2(vec: np.ndarray) -> np.ndarray:
    vec = np.asarray(vec, dtype=float).reshape(-1)
    n = int(np.ceil(np.log2(vec.size)))
    M = 2**n
    if M != vec.size:
        vec = np.pad(vec, (0, M - vec.size), mode="constant")
    return vec

In [20]:
# Build unitary operator for xi
def build_unitary_operator(x: np.ndarray):
    x = np.asarray(x, dtype=complex)

    # correct number of qubits
    n = int(np.ceil(np.log2(x.size)))
    if 2**n != x.size:
        x = np.pad(x, (0, 2**n - x.size), mode="constant")

    # normalize robustly
    norm = np.linalg.norm(x)
    if not np.isclose(norm, 1.0):
        x = x / norm

    # unitary state-prep gate: U |0..0> = |x>
    prep = StatePreparation(x)
    qc = QuantumCircuit(n)
    qc.append(prep, qc.qubits)

    U = Operator(qc).data
    return U

In [21]:
# Test unitary oeprator for an x
random_x = np.random.normal(size=(D))

In [22]:
random_U = build_unitary_operator(random_x)

In [23]:
print(random_U)
conj_transpose = random_U.conj().T
identity = np.identity(random_U.shape[0], dtype=random_U.dtype)
product = random_U @ conj_transpose
tolerance = 1e-8
print("Is random_U a unitary?", np.allclose(product, identity, atol=tolerance))

[[ 2.12855926e-01+1.11022302e-16j -6.24219220e-01+5.96744876e-16j
  -1.58677817e-01+0.00000000e+00j  4.65337024e-01-6.66133815e-16j
  -1.47134949e-01-8.32667268e-17j  4.31486522e-01-3.88578059e-16j
   1.09684766e-01-1.11022302e-16j -3.21660480e-01+4.92661467e-16j]
 [-6.24219220e-01-5.55111512e-16j -2.12855926e-01+3.88578059e-16j
   4.65337024e-01+1.80411242e-16j  1.58677817e-01-4.16333634e-16j
   4.31486522e-01+3.60822483e-16j  1.47134949e-01-2.77555756e-16j
  -3.21660480e-01-9.02056208e-17j -1.09684766e-01+2.22044605e-16j]
 [-7.86083490e-02-6.66133815e-16j  4.85322493e-01-2.56739074e-16j
  -1.05447965e-01-7.21644966e-16j  6.51028423e-01-6.93889390e-16j
   5.43373898e-02+4.44089210e-16j -3.35475275e-01+1.45716772e-16j
   7.28900590e-02+5.55111512e-16j -4.50018168e-01+4.30211422e-16j]
 [ 4.85322493e-01-2.56739074e-16j  7.86083490e-02+1.94289029e-16j
   6.51028423e-01-6.80011603e-16j  1.05447965e-01+1.11022302e-16j
  -3.35475275e-01+1.45716772e-16j -5.43373898e-02-1.66533454e-16j
  -4.50

In [24]:
# Prepare unitary Uxi for each xi
Uxi_list = []
for xi in X:
    Uxi = build_unitary_operator(xi)
    Uxi_list.append(Uxi)

In [25]:
# Compute c = \Sigma_0 m_0
c = Cov0 @ m0

In [26]:
print("c: ", c)

c:  [[4.02531161]
 [4.00456387]
 [3.98726851]
 [4.02616567]
 [3.97898044]
 [3.99989339]]


In [27]:
# Build unitary Uc for c
c = c.reshape(-1)
Uc = build_unitary_operator(c)

In [28]:
# Use an ansatz
SU2_ansatz = efficient_su2(NUM_QUBIT, su2_gates=["rx", "y"], entanglement="linear", reps=REPS)

In [29]:
print(SU2_ansatz)

     ┌──────────┐┌───┐     ┌──────────┐   ┌───┐         
q_0: ┤ Rx(θ[0]) ├┤ Y ├──■──┤ Rx(θ[3]) ├───┤ Y ├─────────
     ├──────────┤├───┤┌─┴─┐└──────────┘┌──┴───┴───┐┌───┐
q_1: ┤ Rx(θ[1]) ├┤ Y ├┤ X ├─────■──────┤ Rx(θ[4]) ├┤ Y ├
     ├──────────┤├───┤└───┘   ┌─┴─┐    ├──────────┤├───┤
q_2: ┤ Rx(θ[2]) ├┤ Y ├────────┤ X ├────┤ Rx(θ[5]) ├┤ Y ├
     └──────────┘└───┘        └───┘    └──────────┘└───┘


In [30]:
# initialize the parameter theta and prepare U(theta) by applying the ansatz parameterized theta on |000...>
theta = SU2_ansatz.parameters
print(theta)

ParameterView([ParameterVectorElement(θ[0]), ParameterVectorElement(θ[1]), ParameterVectorElement(θ[2]), ParameterVectorElement(θ[3]), ParameterVectorElement(θ[4]), ParameterVectorElement(θ[5])])


In [31]:
theta_params = list(SU2_ansatz.parameters)
num_params = len(theta_params)
print(SU2_ansatz)
print("num_params =", num_params)

     ┌──────────┐┌───┐     ┌──────────┐   ┌───┐         
q_0: ┤ Rx(θ[0]) ├┤ Y ├──■──┤ Rx(θ[3]) ├───┤ Y ├─────────
     ├──────────┤├───┤┌─┴─┐└──────────┘┌──┴───┴───┐┌───┐
q_1: ┤ Rx(θ[1]) ├┤ Y ├┤ X ├─────■──────┤ Rx(θ[4]) ├┤ Y ├
     ├──────────┤├───┤└───┘   ┌─┴─┐    ├──────────┤├───┤
q_2: ┤ Rx(θ[2]) ├┤ Y ├────────┤ X ├────┤ Rx(θ[5]) ├┤ Y ├
     └──────────┘└───┘        └───┘    └──────────┘└───┘
num_params = 6


In [32]:
def build_overlap_circuit(Uv_gate, ansatz, theta_bind: dict, num_work_qubits: int):
    """
    Implements:
      H on ancilla
      controlled-Uv on ancilla==0
      controlled-U(theta) on ancilla==1
      H on ancilla
      measure ancilla
    """
    qc = QuantumCircuit(1 + num_work_qubits, 1)
    anc = 0
    work = list(range(1, 1 + num_work_qubits))

    qc.h(anc)

    # control-on-0 for Uv: X-ctrl-X trick
    qc.x(anc)
    qc.append(Uv_gate.control(1), [anc] + work)
    qc.x(anc)

    # control-on-1 for U(theta)
    bound_ansatz = ansatz.assign_parameters(theta_bind, inplace=False)
    qc.append(bound_ansatz.to_gate(label="Uθ").control(1), [anc] + work)

    qc.h(anc)
    qc.measure(anc, 0)
    return qc

In [33]:
def estimate_overlap(Uv_gate, ansatz, theta_vec: np.ndarray, shots: int) -> float:
    theta_bind = {p: float(v) for p, v in zip(theta_params, theta_vec)}
    qc = build_overlap_circuit(Uv_gate, ansatz, theta_bind, NUM_QUBIT)
    # tqc = transpile(qc, backend_qasm, optimization_level=1)
    isa_qc = pm.run(qc)
    # counts = backend_qasm.run(tqc, shots=shots).result().get_counts()
    pub = (isa_qc,)
    job = sampler.run(pub, shots=SHOTS)
    result = job.result()[0]
    counts = result.data.get_counts()
    n0 = counts.get("0", 0)
    p0 = n0 / float(shots)
    s_hat = 2.0 * p0 - 1.0
    return float(np.clip(s_hat, -1.0, 1.0))

In [34]:
theta0 = np.zeros(num_params)
print("Overlap with x0:", estimate_overlap(Uxi_list[0], SU2_ansatz, theta0, shots=SHOTS))
print("Overlap with c :", estimate_overlap(Uc, SU2_ansatz, theta0, shots=SHOTS))

AttributeError: 'numpy.ndarray' object has no attribute 'control'

In [ ]:
sigma_inv_sq = 1.0 / np.diag(Cov0)       # length D
sigma_inv_sq = pad_to_pow2(sigma_inv_sq)  # length 2^d
assert len(sigma_inv_sq) == 2**NUM_QUBIT

In [ ]:
def estimate_d_hat(ansatz, theta_vec: np.ndarray, shots: int) -> float:
    theta_bind = {p: float(v) for p, v in zip(theta_params, theta_vec)}
    qc = QuantumCircuit(NUM_QUBIT, NUM_QUBIT)
    qc.compose(ansatz.assign_parameters(theta_bind, inplace=False), inplace=True)
    qc.measure(range(NUM_QUBIT), range(NUM_QUBIT))
    tqc = pm.run(qc)
    counts = sampler.run(tqc, shots=shots).result().get_counts()
    # Convert bitstring -> index (little-endian consistent with measure order)
    total = 0.0
    for bitstr, cnt in counts.items():
        k = int(bitstr[::-1], 2)
        total += sigma_inv_sq[k] * cnt
    return float(total / float(shots))

In [ ]:
def reduced_objective(theta_vec: np.ndarray, shots: int, batch_size: int, eps: float = 1e-12) -> dict:
    # minibatch indices
    B = np.random.choice(N, size=min(batch_size, N), replace=False)

    # overlaps s_i
    s = np.array([estimate_overlap(Uxi_list[i], SU2_ansatz, theta_vec, shots) for i in B], dtype=float)

    r = np.linalg.norm(X[B, :], axis=1)     # r_i = ||x_i||_2
    yB = y[B].reshape(-1)

    a_hat = float(np.sum((r * s) ** 2))
    c_hat = float(np.sum(yB * (r * s)))

    # prior overlap h, e_hat = ||c|| h
    h_hat = estimate_overlap(Uc, SU2_ansatz, theta_vec, shots)
    e_hat = float(np.linalg.norm(c) * h_hat)

    # diagonal prior quadratic
    d_hat = estimate_d_hat(SU2_ansatz, theta_vec, shots)

    denom = a_hat + d_hat + eps
    L_tilde = -((c_hat + e_hat) ** 2) / denom

    return {
        "L_tilde": float(L_tilde),
        "a_hat": a_hat,
        "c_hat": c_hat,
        "d_hat": d_hat,
        "e_hat": e_hat,
        "h_hat": h_hat,
        "denom": denom,
        "B": B
    }

In [ ]:
# test
theta0 = np.zeros(num_params)
print(reduced_objective(theta0, shots=SHOTS, batch_size=BATCH_SIZE))

In [ ]:
def f(theta_vec):
    return reduced_objective(theta_vec, shots=SHOTS, batch_size=BATCH_SIZE)["L_tilde"]

theta_init = 0.01 * np.random.randn(num_params)

res = minimize(
    fun=f,
    x0=theta_init,
    method="COBYLA",
    options={"maxiter": 100, "disp": True}
)

theta_star = res.x
print("Best L_tilde:", res.fun)

In [ ]:
# Get phi(theta*) exactly using statevector simulator
theta_bind = {p: float(v) for p, v in zip(theta_params, theta_star)}
qc = QuantumCircuit(NUM_QUBIT)
qc.compose(SU2_ansatz.assign_parameters(theta_bind, inplace=False), inplace=True)
tqc = pm.run(qc)
tqc.save_statevector()
sv = backend.run(tqc).result().get_statevector(tqc)

phi = np.real(np.array(sv))   # real ansatz -> real amplitudes
phi = phi / np.linalg.norm(phi)

In [ ]:
# Classical MAP direction (pad to 2^d)
w_dir = (w_star.reshape(-1) / np.linalg.norm(w_star)).astype(float)
w_dir = pad_to_pow2(w_dir)
w_dir = w_dir / np.linalg.norm(w_dir)

cos_sim = float(np.dot(phi, w_dir))
print("Cosine similarity(phi, w*):", cos_sim)